In [22]:
# pip install torch transformers datasets peft accelerate
#df

In [23]:
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import LoraConfig, get_peft_model

In [24]:
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.14.0+cu126
GPU: NVIDIA GeForce GTX 1650


In [25]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [27]:
data = [
    {
        "text": """User: What is Python?
Assistant: Python is a programming language known for being simple, readable, and beginner-friendly."""
    },

    {
        "text": """User: What is machine learning?
Assistant: Machine learning is a method where computers learn patterns from data instead of being explicitly programmed for every rule."""
    },

    {
        "text": """User: What is LoRA?
Assistant: LoRA is a parameter-efficient fine-tuning technique that trains small adapter weights while keeping the original model mostly frozen."""
    },

    {
        "text": """User: What is PEFT?
Assistant: PEFT stands for Parameter-Efficient Fine-Tuning. It allows us to adapt a large model by training only a small number of parameters."""
    },

    {
        "text": """User: What is a neural network?
Assistant: A neural network is a machine learning model made of connected layers that learn patterns from data."""
    },

    {
        "text": """User: Explain AI.
Assistant: Artificial intelligence is the field of building computer systems that can perform tasks that normally require human intelligence."""
    },
]

In [28]:
dataset = Dataset.from_list(data)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 6
})


In [29]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [30]:
model = AutoModelForCausalLM.from_pretrained(
    model_name
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [31]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [32]:
# print the number of model parameters
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params}")
for name, param in model.named_parameters():
    print(name, param.requires_grad)


Total parameters: 1100048384
model.embed_tokens.weight True
model.layers.0.self_attn.q_proj.weight True
model.layers.0.self_attn.k_proj.weight True
model.layers.0.self_attn.v_proj.weight True
model.layers.0.self_attn.o_proj.weight True
model.layers.0.mlp.gate_proj.weight True
model.layers.0.mlp.up_proj.weight True
model.layers.0.mlp.down_proj.weight True
model.layers.0.input_layernorm.weight True
model.layers.0.post_attention_layernorm.weight True
model.layers.1.self_attn.q_proj.weight True
model.layers.1.self_attn.k_proj.weight True
model.layers.1.self_attn.v_proj.weight True
model.layers.1.self_attn.o_proj.weight True
model.layers.1.mlp.gate_proj.weight True
model.layers.1.mlp.up_proj.weight True
model.layers.1.mlp.down_proj.weight True
model.layers.1.input_layernorm.weight True
model.layers.1.post_attention_layernorm.weight True
model.layers.2.self_attn.q_proj.weight True
model.layers.2.self_attn.k_proj.weight True
model.layers.2.self_attn.v_proj.weight True
model.layers.2.self_attn

In [33]:
prompt = """User: What is LoRA in LLM?
Assistant:"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

# Move inputs to GPU if available
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}
    model = model.cuda()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True,
    )

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: What is LoRA in LLM?
Assistant: LoRA in LLM is a LoRaWAN device that is designed to be used in Low Power Wide Area Network (LPWAN) applications. It is a low-power wireless network that delivers high-speed wireless communications over a distance of up to 1 Km. LoRaWAN is a part of the LoRa Alliance and is designed to be


LoRA Config

In [34]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

In [35]:
model = get_peft_model(
    model,
    lora_config
)

In [36]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [37]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [38]:
training_args = TrainingArguments(
    output_dir="./lora-output",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

In [39]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

In [40]:
trainer.train()

Step,Training Loss
1,3.080644
2,2.621904
3,2.548506
4,3.381869
5,2.933942
6,2.519005
7,2.878582
8,2.450953
9,2.871608
10,2.216780


TrainOutput(global_step=10, training_loss=2.750379180908203, metrics={'train_runtime': 37.281, 'train_samples_per_second': 0.805, 'train_steps_per_second': 0.268, 'total_flos': 6462386012160.0, 'train_loss': 2.750379180908203, 'epoch': 5.0})

In [41]:
model.save_pretrained("./my-lora-adapter")

tokenizer.save_pretrained("./my-lora-adapter")

('./my-lora-adapter\\tokenizer_config.json',
 './my-lora-adapter\\chat_template.jinja',
 './my-lora-adapter\\tokenizer.json')

In [42]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

model = PeftModel.from_pretrained(
    base_model,
    "./my-lora-adapter"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [43]:
prompt = """User: What is LoRA in LLM?
Assistant:"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}
    model = model.cuda()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True,
    )

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: What is LoRA in LLM?
Assistant: LoRA in LLM is a low-latency, low-power, and low-cost wireless communication protocol designed for use in IoT devices. It uses short-range radio frequencies (RF) and is ideal for low-power, large-scale IoT applications. The protocol allows for reliable communication between devices, even in harsh weather conditions or in dense environments.
